# Detección de Moniliasis en Cacao con CNNs
### Curso SI3014 — Redes Neuronales y Aprendizaje Profundo

**Autoras:** Mariana Valderrama Castañeda · Sara López Marín · Alexandra Hurtado David

---

En este notebook entrenamos y comparamos tres modelos para clasificar imágenes de mazorcas de cacao como **sanas** o **infectadas con *Monilia roreri***.

Los tres modelos que vamos a construir son:

| Modelo | Descripción |
|--------|-------------|
| **Modelo 1** | CNN construida desde cero (*baseline*) |
| **Modelo 2** | Misma CNN + más augmentación y regularización |
| **Modelo 3** | ResNet18 con Transfer Learning |

La idea es ver cuánto mejora el desempeño a medida que añadimos complejidad y conocimiento previo.


## 0. Instalación de dependencias

Ejecutamos esto primero para asegurarnos de tener todo lo necesario en Colab.

In [ ]:
# Instalamos las librerías que no vienen por defecto en Colab
# torch y torchvision ya están disponibles, pero por si acaso:
!pip install -q torch torchvision
!pip install -q scikit-learn matplotlib seaborn tqdm

# Verificamos que PyTorch reconoce la GPU (si estamos en Colab con GPU activada)
import torch
print(f"PyTorch versión: {torch.__version__}")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


: 

## 1. Descarga del Dataset

El dataset que usamos es el **CocoaMoniliaDataSet** publicado en Zenodo por Alvarado et al. (2026).
Contiene 1,953 imágenes de mazorcas de cacao con 4 clases originales.
Nosotras colapsamos las 3 clases de Monilia en una sola etiqueta `infectada`.

> El dataset se descarga directamente desde Zenodo usando su DOI.


In [ ]:
import os
import zipfile
import requests
from pathlib import Path

# ── Configuración de rutas ──────────────────────────────────────────────────
DATA_DIR = Path("data")           # carpeta raíz donde guardaremos todo
RAW_DIR  = DATA_DIR / "raw"       # imágenes originales sin modificar
DATA_DIR.mkdir(exist_ok=True)
RAW_DIR.mkdir(exist_ok=True)

# ── Descarga desde Zenodo ───────────────────────────────────────────────────
# El record ID de Zenodo corresponde al CocoaMoniliaDataSet
ZENODO_RECORD = "17716661"
ZENODO_URL    = f"https://zenodo.org/api/records/{ZENODO_RECORD}"

print("Consultando metadatos del dataset en Zenodo...")
response = requests.get(ZENODO_URL)
record   = response.json()

# Listamos los archivos disponibles en el registro
print("\nArchivos disponibles:")
for f in record["files"]:
    size_mb = f["size"] / 1e6
    print(f"  {f['key']}  ({size_mb:.1f} MB)")


In [ ]:
# ── Descargamos el archivo principal ────────────────────────────────────────
# Ajusta el nombre del archivo según lo que aparezca en la celda anterior
FILE_KEY = record["files"][0]["key"]   # tomamos el primero por defecto
FILE_URL = record["files"][0]["links"]["self"]

dest = RAW_DIR / FILE_KEY

if dest.exists():
    print(f"El archivo {FILE_KEY} ya existe, omitimos la descarga.")
else:
    print(f"Descargando {FILE_KEY}...")
    with requests.get(FILE_URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
                downloaded += len(chunk)
                pct = downloaded / total * 100 if total else 0
                print(f"  {pct:.1f}%", end="\r")
    print(f"\nDescarga completa: {dest}")

# ── Descomprimimos si es un ZIP ─────────────────────────────────────────────
if str(dest).endswith(".zip"):
    print("Descomprimiendo...")
    with zipfile.ZipFile(dest, "r") as z:
        z.extractall(RAW_DIR)
    print("Listo.")


In [ ]:
# ── Exploramos la estructura de carpetas del dataset ────────────────────────
# Queremos entender cómo están organizadas las imágenes antes de cargarlas.
# Lo típico en datasets de visión es: una carpeta por clase.

for root, dirs, files in os.walk(RAW_DIR):
    level = root.replace(str(RAW_DIR), "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    n_files = len([f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    if n_files > 0:
        print(f"{indent} {folder}/  → {n_files} imágenes")
    elif dirs:
        print(f"{indent} {folder}/")


## 2. Reorganización Binaria del Dataset

El dataset original tiene **4 clases**: `h0` (sana), `m1`, `m2`, `m3` (infectadas).
Nosotras las colapsamos en **2 clases** para hacer clasificación binaria:

- `sana`     ← imágenes de la carpeta `h0`
- `infectada` ← imágenes de las carpetas `m1`, `m2`, `m3`

Creamos una nueva estructura de carpetas limpia para que PyTorch la pueda leer directamente con `ImageFolder`.


In [ ]:
import shutil

# ── Definimos el mapeo de clases originales a binarias ──────────────────────
# Ajusta los nombres de carpeta según lo que viste en la celda anterior
CLASE_SANA      = ["h0"]             # carpetas que corresponden a "sana"
CLASES_INFECTADA = ["m1", "m2", "m3"] # carpetas que corresponden a "infectada"

BINARY_DIR = DATA_DIR / "binary"     # aquí quedará la estructura binaria
for split in ["train", "val", "test"]:
    (BINARY_DIR / split / "sana").mkdir(parents=True, exist_ok=True)
    (BINARY_DIR / split / "infectada").mkdir(parents=True, exist_ok=True)

# ── Recopilamos todas las rutas e imágenes ──────────────────────────────────
from sklearn.model_selection import train_test_split

all_images = []   # lista de (ruta_imagen, etiqueta_binaria)

for root, dirs, files in os.walk(RAW_DIR):
    carpeta = os.path.basename(root)
    for fname in files:
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        ruta = os.path.join(root, fname)
        if carpeta in CLASE_SANA:
            all_images.append((ruta, "sana"))
        elif carpeta in CLASES_INFECTADA:
            all_images.append((ruta, "infectada"))

print(f"Total de imágenes encontradas: {len(all_images)}")
sanas     = sum(1 for _, y in all_images if y == "sana")
infectadas = sum(1 for _, y in all_images if y == "infectada")
print(f"  Sanas:      {sanas}")
print(f"  Infectadas: {infectadas}")


In [ ]:
# ── Dividimos en train / val / test con estratificación ─────────────────────
# La estratificación garantiza que la proporción sana/infectada
# sea la misma en los tres conjuntos. Esto es importante cuando
# las clases no están perfectamente balanceadas.

rutas  = [r for r, _ in all_images]
labels = [y for _, y in all_images]

# Primero separamos el 70% para entrenamiento
train_r, temp_r, train_y, temp_y = train_test_split(
    rutas, labels, test_size=0.30, stratify=labels, random_state=42
)

# Del 30% restante, la mitad es validación y la otra mitad es prueba (15% / 15%)
val_r, test_r, val_y, test_y = train_test_split(
    temp_r, temp_y, test_size=0.50, stratify=temp_y, random_state=42
)

print(f"Entrenamiento: {len(train_r)} imágenes")
print(f"Validación:    {len(val_r)} imágenes")
print(f"Prueba:        {len(test_r)} imágenes")

# ── Copiamos las imágenes a las carpetas correspondientes ───────────────────
def copiar_imagenes(rutas, etiquetas, split_name):
    for ruta, etiqueta in zip(rutas, etiquetas):
        fname = os.path.basename(ruta)
        dest  = BINARY_DIR / split_name / etiqueta / fname
        if not dest.exists():
            shutil.copy2(ruta, dest)

copiar_imagenes(train_r, train_y, "train")
copiar_imagenes(val_r,   val_y,   "val")
copiar_imagenes(test_r,  test_y,  "test")
print("\n Dataset reorganizado correctamente.")


## 3. Análisis Exploratorio (EDA)

Antes de entrenar cualquier modelo, es buena práctica explorar el dataset:
- ¿Cuántas imágenes hay por clase?
- ¿Están balanceadas?
- ¿Cómo se ven las imágenes?
- ¿Cuál es la distribución de píxeles?


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from collections import Counter

# ── Distribución de clases por split ────────────────────────────────────────
splits = ["train", "val", "test"]
clases = ["sana", "infectada"]

conteos = {split: {clase: len(list((BINARY_DIR / split / clase).glob("*")))
                   for clase in clases}
           for split in splits}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colores = ["#4CAF50", "#F44336"]   # verde para sana, rojo para infectada

for ax, split in zip(axes, splits):
    valores = [conteos[split][c] for c in clases]
    bars = ax.bar(clases, valores, color=colores, edgecolor="white", width=0.5)
    ax.set_title(f"{split.capitalize()}  (n={sum(valores)})", fontsize=13)
    ax.set_ylabel("Número de imágenes")
    ax.set_ylim(0, max(valores) * 1.25)
    # Añadimos el número encima de cada barra
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
                str(val), ha="center", fontsize=11, fontweight="bold")

plt.suptitle("Distribución de clases por conjunto", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("eda_distribucion.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada: eda_distribucion.png")


In [ ]:
# ── Visualizamos ejemplos de cada clase ─────────────────────────────────────
# Ver las imágenes nos ayuda a entender qué tan difícil es la tarea visualmente.

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle("Ejemplos del dataset — fila superior: sanas · fila inferior: infectadas",
             fontsize=13)

for col, clase in enumerate(["sana", "infectada"] * 5):
    fila   = 0 if clase == "sana" else 1
    col_ax = col if col < 5 else col - 5
    ax     = axes[fila][col_ax]

    # Tomamos una imagen aleatoria de la carpeta de entrenamiento
    carpeta = BINARY_DIR / "train" / clase
    img_paths = list(carpeta.glob("*"))
    img_path  = img_paths[col_ax * 3 % len(img_paths)]  # variamos el índice

    img = mpimg.imread(str(img_path))
    ax.imshow(img)
    ax.axis("off")
    ax.set_title(clase, fontsize=10,
                 color="#4CAF50" if clase == "sana" else "#F44336")

plt.tight_layout()
plt.savefig("eda_ejemplos.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figura guardada: eda_ejemplos.png")


In [ ]:
# ── Distribución de píxeles por canal (R, G, B) ─────────────────────────────
# Muestreamos 200 imágenes del set de entrenamiento para ver la distribución
# de intensidades. Esto nos ayuda a decidir cómo normalizar.

sample_paths = list((BINARY_DIR / "train" / "sana").glob("*"))[:100] + \
               list((BINARY_DIR / "train" / "infectada").glob("*"))[:100]

r_vals, g_vals, b_vals = [], [], []

for p in sample_paths:
    img = mpimg.imread(str(p))
    if img.max() > 1:           # imágenes en rango [0, 255]
        img = img / 255.0
    if img.ndim == 3 and img.shape[2] >= 3:
        r_vals.extend(img[:, :, 0].flatten().tolist())
        g_vals.extend(img[:, :, 1].flatten().tolist())
        b_vals.extend(img[:, :, 2].flatten().tolist())

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(r_vals, bins=50, color="#E53935", alpha=0.6, label="Canal R", density=True)
ax.hist(g_vals, bins=50, color="#43A047", alpha=0.6, label="Canal G", density=True)
ax.hist(b_vals, bins=50, color="#1E88E5", alpha=0.6, label="Canal B", density=True)
ax.set_xlabel("Intensidad de píxel (normalizada)")
ax.set_ylabel("Densidad")
ax.set_title("Distribución de intensidades por canal RGB (muestra de 200 imágenes)")
ax.legend()
plt.tight_layout()
plt.savefig("eda_pixeles.png", dpi=120, bbox_inches="tight")
plt.show()


## 4. Preprocesamiento y DataLoaders

Definimos las transformaciones que se aplican a las imágenes antes de pasarlas al modelo.

- **Entrenamiento:** redimensionamos + augmentación + normalización
- **Validación y prueba:** solo redimensionamos + normalizamos (sin augmentación, para evaluar limpiamente)


In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── Parámetros globales ──────────────────────────────────────────────────────
IMG_SIZE   = 224    # tamaño al que redimensionamos todas las imágenes
BATCH_SIZE = 32     # número de imágenes por batch
NUM_WORKERS = 2     # hilos paralelos para cargar datos (0 si hay problemas en Colab)

# Media y desviación estándar de ImageNet.
# Las usamos porque nuestros modelos con transfer learning esperan
# que las imágenes estén normalizadas con estos valores.
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Transformaciones ─────────────────────────────────────────────────────────
# Para entrenamiento aplicamos augmentación para que el modelo vea
# variaciones artificiales y no memorice las imágenes exactas.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),                          # flip horizontal
    transforms.RandomVerticalFlip(),                            # flip vertical
    transforms.RandomRotation(degrees=30),                      # rotación aleatoria
    transforms.ColorJitter(brightness=0.3, contrast=0.3,       # cambios de brillo/contraste
                           saturation=0.2),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),   # recorte aleatorio
    transforms.ToTensor(),                                      # convierte a tensor [0,1]
    transforms.Normalize(MEAN, STD),                            # normalización ImageNet
])

# Para validación y prueba NO aplicamos augmentación.
# Solo redimensionamos y normalizamos para evaluar de forma justa.
val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Datasets y DataLoaders ───────────────────────────────────────────────────
# ImageFolder asume que la estructura de carpetas es:
#   split/
#     clase_1/  ← imágenes de la clase 1
#     clase_2/  ← imágenes de la clase 2

train_dataset = datasets.ImageFolder(BINARY_DIR / "train", transform=train_transform)
val_dataset   = datasets.ImageFolder(BINARY_DIR / "val",   transform=val_test_transform)
test_dataset  = datasets.ImageFolder(BINARY_DIR / "test",  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Verificamos que las clases se asignaron correctamente
print("Clases detectadas:", train_dataset.classes)
print(f"  0 → {train_dataset.classes[0]}  |  1 → {train_dataset.classes[1]}")
print(f"\nTamaños:")
print(f"  Train : {len(train_dataset)} imágenes  ({len(train_loader)} batches)")
print(f"  Val   : {len(val_dataset)} imágenes  ({len(val_loader)} batches)")
print(f"  Test  : {len(test_dataset)} imágenes  ({len(test_loader)} batches)")


In [ ]:
# ── Verificamos que un batch se ve bien ─────────────────────────────────────
# Esta celda es un sanity-check: si algo falló en el preprocesamiento
# lo veríamos aquí con un error.

import torchvision

imagenes, etiquetas = next(iter(train_loader))
print(f"Shape del batch: {imagenes.shape}")   # debe ser [32, 3, 224, 224]
print(f"Etiquetas: {etiquetas[:8].tolist()}")  # 0=sana, 1=infectada

# Desnormalizamos para visualizar
def desnormalizar(tensor):
    mean = torch.tensor(MEAN).view(3, 1, 1)
    std  = torch.tensor(STD).view(3, 1, 1)
    return torch.clamp(tensor * std + mean, 0, 1)

grid = torchvision.utils.make_grid(
    [desnormalizar(imagenes[i]) for i in range(8)], nrow=8
)
plt.figure(figsize=(16, 3))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("Primeras 8 imágenes del primer batch (desnormalizadas)")
plt.axis("off")
plt.tight_layout()
plt.show()


## 5. Funciones de Entrenamiento y Evaluación

Definimos funciones reutilizables para entrenar y evaluar los tres modelos.
Así evitamos repetir código y es más fácil comparar resultados.


In [ ]:
from sklearn.metrics import f1_score, recall_score, accuracy_score, roc_auc_score
from tqdm.notebook import tqdm

# ── Dispositivo ──────────────────────────────────────────────────────────────
# Usamos GPU si está disponible, si no, CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {DEVICE}")


def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    """
    Entrena el modelo por una época completa.
    Retorna la pérdida promedio de la época.
    """
    modelo.train()  # modo entrenamiento: activa Dropout y BatchNorm
    perdida_total = 0.0

    for imagenes, etiquetas in loader:
        imagenes = imagenes.to(DEVICE)
        etiquetas = etiquetas.to(DEVICE)

        optimizador.zero_grad()          # limpiamos gradientes del paso anterior
        salidas = modelo(imagenes)       # forward pass
        perdida = criterio(salidas, etiquetas)  # calculamos la pérdida
        perdida.backward()               # backpropagation
        optimizador.step()               # actualizamos los pesos

        perdida_total += perdida.item() * imagenes.size(0)

    return perdida_total / len(loader.dataset)


def evaluar(modelo, loader):
    """
    Evalúa el modelo sin actualizar pesos (modo inferencia).
    Retorna: pérdida, accuracy, f1, recall, auc
    """
    modelo.eval()  # modo evaluación: desactiva Dropout
    perdida_total = 0.0
    criterio = torch.nn.CrossEntropyLoss()

    todas_preds   = []
    todas_labels  = []
    todas_probs   = []

    with torch.no_grad():  # no calculamos gradientes — ahorra memoria
        for imagenes, etiquetas in loader:
            imagenes  = imagenes.to(DEVICE)
            etiquetas = etiquetas.to(DEVICE)

            salidas = modelo(imagenes)
            perdida = criterio(salidas, etiquetas)
            perdida_total += perdida.item() * imagenes.size(0)

            # Convertimos logits a probabilidades con Softmax
            probs = torch.softmax(salidas, dim=1)[:, 1]  # prob de clase infectada
            preds = salidas.argmax(dim=1)

            todas_preds.extend(preds.cpu().numpy())
            todas_labels.extend(etiquetas.cpu().numpy())
            todas_probs.extend(probs.cpu().numpy())

    perdida_prom = perdida_total / len(loader.dataset)
    acc    = accuracy_score(todas_labels, todas_preds)
    f1     = f1_score(todas_labels, todas_preds, average="macro")
    recall = recall_score(todas_labels, todas_preds, pos_label=1)  # recall infectada
    auc    = roc_auc_score(todas_labels, todas_probs)

    return perdida_prom, acc, f1, recall, auc


def ciclo_entrenamiento(modelo, train_loader, val_loader, optimizador,
                         scheduler, n_epocas=50, paciencia=10,
                         nombre_modelo="modelo"):
    """
    Ciclo completo de entrenamiento con early stopping.

    Early stopping: si el F1 de validación no mejora en `paciencia` épocas,
    detenemos el entrenamiento para evitar overfitting.
    """
    criterio = torch.nn.CrossEntropyLoss()
    modelo   = modelo.to(DEVICE)

    historial = {
        "train_loss": [], "val_loss": [],
        "val_acc": [], "val_f1": [], "val_recall": [], "val_auc": []
    }

    mejor_f1     = 0.0
    epocas_sin_mejora = 0
    mejor_estado = None

    for epoca in range(1, n_epocas + 1):
        train_loss = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
        val_loss, val_acc, val_f1, val_recall, val_auc = evaluar(modelo, val_loader)

        # El scheduler ajusta el LR si la val_loss no mejora
        scheduler.step(val_loss)

        # Guardamos métricas del historial
        historial["train_loss"].append(train_loss)
        historial["val_loss"].append(val_loss)
        historial["val_acc"].append(val_acc)
        historial["val_f1"].append(val_f1)
        historial["val_recall"].append(val_recall)
        historial["val_auc"].append(val_auc)

        print(f"Época {epoca:3d}/{n_epocas}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"F1={val_f1:.4f}  Recall={val_recall:.4f}  AUC={val_auc:.4f}")

        # ── Early stopping ──────────────────────────────────────────────────
        if val_f1 > mejor_f1:
            mejor_f1     = val_f1
            epocas_sin_mejora = 0
            # Guardamos el mejor estado del modelo
            mejor_estado = {k: v.cpu().clone()
                            for k, v in modelo.state_dict().items()}
            torch.save(mejor_estado, f"{nombre_modelo}_best.pt")
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= paciencia:
                print(f"\n⏹ Early stopping en época {epoca}. Mejor F1: {mejor_f1:.4f}")
                break

    # Cargamos los pesos del mejor epoch antes de retornar
    if mejor_estado is not None:
        modelo.load_state_dict({k: v.to(DEVICE) for k, v in mejor_estado.items()})

    return modelo, historial


def graficar_historial(historial, titulo=""):
    """Grafica las curvas de pérdida y F1 durante el entrenamiento."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    epocas = range(1, len(historial["train_loss"]) + 1)

    # Curva de pérdida
    axes[0].plot(epocas, historial["train_loss"], label="Train loss", color="#1E88E5")
    axes[0].plot(epocas, historial["val_loss"],   label="Val loss",   color="#F44336")
    axes[0].set_xlabel("Época")
    axes[0].set_ylabel("CrossEntropyLoss")
    axes[0].set_title("Pérdida durante el entrenamiento")
    axes[0].legend()

    # Curva de F1
    axes[1].plot(epocas, historial["val_f1"],     label="Val F1",     color="#43A047")
    axes[1].plot(epocas, historial["val_recall"], label="Val Recall",  color="#FB8C00")
    axes[1].set_xlabel("Época")
    axes[1].set_ylabel("Métrica")
    axes[1].set_title("F1 y Recall en validación")
    axes[1].legend()

    plt.suptitle(titulo, fontsize=13)
    plt.tight_layout()
    plt.savefig(f"{titulo.replace(' ', '_')}_curvas.png", dpi=120, bbox_inches="tight")
    plt.show()


## 6. Modelo 1 — CNN Baseline (from scratch)

Construimos una CNN sencilla desde cero. La llamamos *baseline* porque
nos sirve de punto de comparación: si los modelos siguientes no la superan,
algo está mal.

La arquitectura tiene 4 bloques convolucionales, cada uno con:
- **Conv2D** → extrae características locales
- **BatchNorm** → estabiliza el entrenamiento normalizando activaciones
- **ReLU** → introduce no-linealidad
- **MaxPool** → reduce la resolución espacial (y el costo computacional)

Al final, una capa **Dense** toma las características y produce la clasificación.


In [ ]:
import torch.nn as nn

class CNNBaseline(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNBaseline, self).__init__()

        # ── Bloques convolucionales ──────────────────────────────────────────
        # Cada bloque extrae características cada vez más abstractas.
        # Los filtros van de 32 → 64 → 128 → 256, capturando
        # desde bordes simples hasta texturas complejas.
        self.features = nn.Sequential(

            # Bloque 1: entrada 224×224×3 → salida 112×112×32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Bloque 2: 112×112×32 → 56×56×64
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Bloque 3: 56×56×64 → 28×28×128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            # Bloque 4: 28×28×128 → 14×14×256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )

        # GlobalAveragePooling: colapsa cada mapa de características a un
        # solo número (promedio), independiente del tamaño de entrada.
        # Salida: 256 valores
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Clasificador ─────────────────────────────────────────────────────
        self.classifier = nn.Sequential(
            nn.Flatten(),                   # 256×1×1 → vector de 256
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),                # apagamos el 50% de neuronas al azar
            nn.Linear(256, num_classes),    # salida: 2 logits (sana / infectada)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


# ── Verificamos que el modelo funciona con un batch de prueba ────────────────
modelo1 = CNNBaseline(num_classes=2)
dummy   = torch.zeros(4, 3, 224, 224)        # batch falso de 4 imágenes
salida  = modelo1(dummy)
print(f"Shape de salida: {salida.shape}")     # esperamos [4, 2]

n_params = sum(p.numel() for p in modelo1.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {n_params:,}")


In [ ]:
# ── Entrenamiento del Modelo 1 ───────────────────────────────────────────────
import torch.optim as optim

modelo1 = CNNBaseline(num_classes=2)

optimizador1 = optim.Adam(modelo1.parameters(), lr=1e-3)

# ReduceLROnPlateau: reduce el learning rate cuando la val_loss
# deja de mejorar durante `patience` épocas.
scheduler1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizador1, mode="min", patience=5, factor=0.5, verbose=True
)

print("=" * 60)
print("Entrenando Modelo 1 — CNN Baseline")
print("=" * 60)

modelo1, hist1 = ciclo_entrenamiento(
    modelo1, train_loader, val_loader,
    optimizador1, scheduler1,
    n_epocas=50, paciencia=10,
    nombre_modelo="modelo1_baseline"
)


In [ ]:
graficar_historial(hist1, titulo="Modelo 1 - CNN Baseline")

# Evaluamos en el conjunto de prueba (solo al final, nunca durante el entrenamiento)
_, acc1, f1_1, rec1, auc1 = evaluar(modelo1, test_loader)
print(f"\n Resultados Modelo 1 en TEST:")
print(f"   Accuracy : {acc1:.4f}")
print(f"   F1-score : {f1_1:.4f}")
print(f"   Recall   : {rec1:.4f}")
print(f"   AUC-ROC  : {auc1:.4f}")


## 7. Modelo 2 — CNN con más Regularización

Mismo backbone que el Modelo 1, pero añadimos:
- Una capa Dense adicional en el clasificador
- Doble Dropout (0.5 y 0.3)
- Weight decay en el optimizador

El objetivo es ver si regularizar mejor el mismo modelo reduce el overfitting
y mejora la generalización.


In [ ]:
class CNNRegularizada(nn.Module):
    def __init__(self, num_classes=2):
        super(CNNRegularizada, self).__init__()

        # El backbone es idéntico al Modelo 1
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)

        # ── Clasificador más profundo con más regularización ─────────────────
        # Añadimos una capa intermedia Dense(128) y dos capas de Dropout.
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),            # dropout más agresivo en la primera capa
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),            # dropout más suave en la segunda
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.gap(x)
        x = self.classifier(x)
        return x


modelo2   = CNNRegularizada(num_classes=2)

# weight_decay añade una penalización L2 a los pesos grandes,
# lo que ayuda a que el modelo no se sobreajuste a los datos de entrenamiento.
optimizador2 = optim.Adam(modelo2.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler2   = optim.lr_scheduler.ReduceLROnPlateau(
    optimizador2, mode="min", patience=5, factor=0.5, verbose=True
)

print("=" * 60)
print("Entrenando Modelo 2 — CNN Regularizada")
print("=" * 60)

modelo2, hist2 = ciclo_entrenamiento(
    modelo2, train_loader, val_loader,
    optimizador2, scheduler2,
    n_epocas=50, paciencia=10,
    nombre_modelo="modelo2_regularizado"
)


In [ ]:
graficar_historial(hist2, titulo="Modelo 2 - CNN Regularizada")

_, acc2, f1_2, rec2, auc2 = evaluar(modelo2, test_loader)
print(f"\n Resultados Modelo 2 en TEST:")
print(f"   Accuracy : {acc2:.4f}")
print(f"   F1-score : {f1_2:.4f}")
print(f"   Recall   : {rec2:.4f}")
print(f"   AUC-ROC  : {auc2:.4f}")


## 8. Modelo 3 — Transfer Learning con ResNet18

Usamos **ResNet18** preentrenada en ImageNet como punto de partida.
La idea es que los filtros aprendidos en millones de imágenes ya capturan
bordes, texturas y formas útiles para cualquier tarea visual, incluyendo
la detección de enfermedades en plantas.

Entrenamos en **dos fases**:

1. **Fase 1 (congelado):** congelamos todos los pesos del backbone y solo entrenamos el clasificador que añadimos al final. Esto evita destruir los pesos preentrenados.
2. **Fase 2 (fine-tuning):** descongelamos la última capa del backbone (`layer4`) y entrenamos con un LR muy pequeño para afinar los pesos al dominio del cacao.


In [ ]:
from torchvision import models

# ── Cargamos ResNet18 con pesos preentrenados en ImageNet ────────────────────
resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Congelamos todos los parámetros del backbone.
# Esto significa que durante la Fase 1 solo se actualizan los pesos
# del clasificador que añadiremos al final.
for param in resnet.parameters():
    param.requires_grad = False

# Reemplazamos la capa final de ResNet18.
# La original clasifica 1000 clases (ImageNet); nosotras necesitamos 2.
n_features = resnet.fc.in_features   # número de entradas a la capa original (512)
resnet.fc = nn.Sequential(
    nn.Linear(n_features, 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.4),
    nn.Linear(256, 2),               # salida: 2 clases
)

modelo3 = resnet

# Verificamos cuántos parámetros son entrenables en esta fase
n_entrenable = sum(p.numel() for p in modelo3.parameters() if p.requires_grad)
n_total      = sum(p.numel() for p in modelo3.parameters())
print(f"Parámetros entrenables (Fase 1): {n_entrenable:,} / {n_total:,}")


In [ ]:
# ── FASE 1: entrenamos solo el clasificador ──────────────────────────────────
# LR más bajo que en los modelos anteriores porque los pesos del backbone
# ya están bien inicializados y no queremos modificarlos.

optimizador3_fase1 = optim.Adam(
    filter(lambda p: p.requires_grad, modelo3.parameters()),
    lr=1e-4, weight_decay=1e-4
)
scheduler3_fase1 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizador3_fase1, mode="min", patience=5, factor=0.5, verbose=True
)

print("=" * 60)
print("Modelo 3 — Fase 1: solo clasificador (backbone congelado)")
print("=" * 60)

modelo3, hist3_fase1 = ciclo_entrenamiento(
    modelo3, train_loader, val_loader,
    optimizador3_fase1, scheduler3_fase1,
    n_epocas=20, paciencia=7,
    nombre_modelo="modelo3_fase1"
)


In [ ]:
# ── FASE 2: fine-tuning — descongelamos layer4 del backbone ─────────────────
# layer4 es el último bloque residual de ResNet18.
# Al descongelarlo, le permitimos al modelo adaptar estas representaciones
# al dominio específico de las mazorcas de cacao.

for param in modelo3.layer4.parameters():
    param.requires_grad = True

n_entrenable = sum(p.numel() for p in modelo3.parameters() if p.requires_grad)
print(f"Parámetros entrenables (Fase 2): {n_entrenable:,}")

# LR más pequeño en la Fase 2 para no dañar los pesos preentrenados
optimizador3_fase2 = optim.Adam(
    filter(lambda p: p.requires_grad, modelo3.parameters()),
    lr=1e-5, weight_decay=1e-4
)
scheduler3_fase2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizador3_fase2, mode="min", patience=5, factor=0.5, verbose=True
)

print("=" * 60)
print("Modelo 3 — Fase 2: fine-tuning de layer4")
print("=" * 60)

modelo3, hist3_fase2 = ciclo_entrenamiento(
    modelo3, train_loader, val_loader,
    optimizador3_fase2, scheduler3_fase2,
    n_epocas=20, paciencia=7,
    nombre_modelo="modelo3_fase2"
)


In [ ]:
# Unimos el historial de las dos fases para graficar juntas
hist3_completo = {k: hist3_fase1[k] + hist3_fase2[k] for k in hist3_fase1}
graficar_historial(hist3_completo, titulo="Modelo 3 - ResNet18 Transfer Learning")

_, acc3, f1_3, rec3, auc3 = evaluar(modelo3, test_loader)
print(f"\n Resultados Modelo 3 en TEST:")
print(f"   Accuracy : {acc3:.4f}")
print(f"   F1-score : {f1_3:.4f}")
print(f"   Recall   : {rec3:.4f}")
print(f"   AUC-ROC  : {auc3:.4f}")


## 9. Comparación Final de los Tres Modelos

Comparamos los resultados de los tres modelos sobre el conjunto de **prueba**
(que no se usó en ningún momento durante el entrenamiento ni la selección de modelos).


In [ ]:
import pandas as pd

# ── Tabla de resultados ──────────────────────────────────────────────────────
resultados = pd.DataFrame({
    "Modelo": [
        "Modelo 1 — CNN Baseline",
        "Modelo 2 — CNN Regularizada",
        "Modelo 3 — ResNet18 TL"
    ],
    "Accuracy": [acc1, acc2, acc3],
    "F1-score": [f1_1, f1_2, f1_3],
    "Recall (infectada)": [rec1, rec2, rec3],
    "AUC-ROC": [auc1, auc2, auc3],
})

# Resaltamos el mejor valor de cada columna
resultados_fmt = resultados.style.highlight_max(
    subset=["Accuracy", "F1-score", "Recall (infectada)", "AUC-ROC"],
    color="lightgreen"
).format({
    "Accuracy": "{:.4f}", "F1-score": "{:.4f}",
    "Recall (infectada)": "{:.4f}", "AUC-ROC": "{:.4f}"
})
display(resultados_fmt)


In [ ]:
# ── Gráfica comparativa ──────────────────────────────────────────────────────
metricas = ["Accuracy", "F1-score", "Recall (infectada)", "AUC-ROC"]
modelos  = ["Baseline", "CNN Reg.", "ResNet18 TL"]
colores  = ["#1E88E5", "#43A047", "#FB8C00"]

x = np.arange(len(metricas))
ancho = 0.25

fig, ax = plt.subplots(figsize=(12, 5))

for i, (modelo, color) in enumerate(zip(modelos, colores)):
    valores = resultados.iloc[i][metricas].values.astype(float)
    bars = ax.bar(x + i * ancho, valores, ancho, label=modelo,
                  color=color, alpha=0.85, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f"{bar.get_height():.3f}",
                ha="center", va="bottom", fontsize=9)

ax.set_xticks(x + ancho)
ax.set_xticklabels(metricas, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel("Valor de la métrica")
ax.set_title("Comparación de los tres modelos en el conjunto de prueba", fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("comparacion_modelos.png", dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ── Análisis de errores: matriz de confusión ─────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
nombres_modelos = ["CNN Baseline", "CNN Regularizada", "ResNet18 TL"]

for ax, modelo, nombre in zip(axes, [modelo1, modelo2, modelo3], nombres_modelos):
    modelo.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, ys in test_loader:
            salidas = modelo(imgs.to(DEVICE))
            preds.extend(salidas.argmax(dim=1).cpu().numpy())
            labels.extend(ys.numpy())

    cm = confusion_matrix(labels, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=["sana", "infectada"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(nombre, fontsize=12)

plt.suptitle("Matrices de confusión — conjunto de prueba", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("matrices_confusion.png", dpi=120, bbox_inches="tight")
plt.show()


## 10. Conclusiones

Con base en los resultados obtenidos, podemos concluir:

- **Modelo 1 (Baseline):** establece el piso de referencia. Cualquier mejora de los modelos siguientes es directamente atribuible a las técnicas añadidas.
- **Modelo 2 (Regularización):** el efecto de la regularización adicional se refleja principalmente en la reducción del gap entre train y val loss (menos overfitting).
- **Modelo 3 (Transfer Learning):** al aprovechar representaciones visuales preaprendidas, ResNet18 logra mayor F1 y Recall con menos épocas de entrenamiento, lo cual es especialmente útil cuando el dataset es pequeño (~1,953 imágenes).

El **Recall sobre la clase infectada** es la métrica más crítica en este problema: un falso negativo (mazorca enferma clasificada como sana) tiene consecuencias reales para el agricultor.

---

## Referencias

1. Alvarado et al. (2026). CocoaMoniliaDataSet. *Data in Brief, 64*, 112447. https://doi.org/10.1016/j.dib.2025.112447
2. He et al. (2016). Deep Residual Learning for Image Recognition. *CVPR 2016*. https://arxiv.org/abs/1512.03385
3. Saleem et al. (2019). Plant Disease Detection by Deep Learning. *Plants, 8*(11), 468.
